In [ ]:
!pip install -q openai

In [ ]:
import base64

from IPython.display import Audio
from pathlib import Path

def print_completion(completion):
    print(f"Completion id: {completion.id}")
    print(f"Input tokens: {completion.usage.prompt_tokens}; Output tokens: {completion.usage.completion_tokens}")
    print()

    if len(completion.choices) > 1:
        raise Exception("Expected a single choice.")

    message = completion.choices[0].message
    if message is None:
        raise Exception("Expected a single message")

    if message.content is not None:
        print(message.content)

def display_audio(path_to_file: Path):
    display(Audio(path_to_file))

In [ ]:
from google.colab import userdata
from openai import OpenAI

api_key = userdata.get("OPENROUTER_API_KEY")
client = OpenAI(api_key=api_key, base_url="https://openrouter.ai/api/v1")

In [ ]:
def transcribe(path_to_file: Path):
    with path_to_file.open(mode="rb") as file:
        file_bytes = file.read()

    return client.chat.completions.create(
        model="mistralai/voxtral-small-24b-2507",
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": "Transcribe this recording faithfully in its original language. Preserve punctuation and formatting where natural. Return only the transcript."
                    },
                    {
                        "type": "input_audio",
                        "input_audio": {
                            "data": base64.b64encode(file_bytes).decode('utf-8'),
                            "format": path_to_file.suffix
                        }
                    }
                ]
            }
        ]
    )

## Speech to Text

The `career_center_softuni.mp3` represents the audio from [this video](https://youtu.be/wMdE2GjBw-A).

In [ ]:
PATH_TO_CAREER_CENTER_AUDIO = Path("/content/career_center_softuni.mp3")
display_audio(PATH_TO_CAREER_CENTER_AUDIO)

In [ ]:
career_center_transcribe_completion = transcribe(PATH_TO_CAREER_CENTER_AUDIO)

In [ ]:
print_completion(career_center_transcribe_completion)